In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. Data Loading and Preprocessing ---

# Load the NEW expanded dataset
try:
    df = pd.read_csv('Expanded_Flexibility_Recommendations.csv') # <--- CHANGED FILENAME HERE
except FileNotFoundError:
    print("Error: 'Expanded_Flexibility_Recommendations.csv' not found.")
    print("Please ensure the file is in the correct directory or upload it.")
    exit()

# Identify Features (X) and Target (y)
X = df[['Age Group', 'Medical Condition', 'Flexibility Level', 'Pain Level']]
y = df['Suggested Category']

# One-Hot Encode Features
encoder_X = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder_X.fit_transform(X)
X_encoded_df = pd.DataFrame(X_encoded, columns=encoder_X.get_feature_names_out(X.columns))

# Label Encode Target
encoder_y = LabelEncoder()
y_encoded = encoder_y.fit_transform(y)

# Splitting the Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded_df, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Data preparation complete. Shapes of training and testing sets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print("\n" + "="*50 + "\n")

# --- 2. Hyperparameter Tuning using GridSearchCV ---

print("Starting Hyperparameter Tuning for RandomForestClassifier...")

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# Create a GridSearchCV object
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=5,
                           n_jobs=-1,
                           verbose=2,
                           scoring='accuracy')

# Fit GridSearchCV to the training data to find the best hyperparameters
grid_search.fit(X_train, y_train)

print("\nHyperparameter tuning complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")
print("\n" + "="*50 + "\n")

# --- 3. Model Training (with best parameters) and Evaluation ---

print("Evaluating the model with best parameters on the test set...")

best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred_best)
print(f"Test Accuracy with Best Model: {test_accuracy:.4f}")

print("\nClassification Report with Best Model:")
print(classification_report(y_test, y_pred_best, target_names=encoder_y.classes_))

print("\nConfusion Matrix with Best Model:")
print(confusion_matrix(y_test, y_pred_best))
print("\n" + "="*50 + "\n")

# --- 4. Saving the Best Model and Encoders ---

print("Saving the best model and encoders for future use...")

model_filename = 'random_forest_model_tuned.pkl'
joblib.dump(best_model, model_filename)
joblib.dump(encoder_X, 'onehot_encoder_X.pkl')
joblib.dump(encoder_y, 'label_encoder_y.pkl')

try:
    df_examples = pd.read_csv('Exercise_Categories_Examples.csv') # This file is still the same
    category_to_examples = dict(zip(df_examples['Category'], df_examples['Examples']))
    joblib.dump(category_to_examples, 'category_examples_map.pkl')
except FileNotFoundError:
    print("Warning: 'Exercise_Categories_Examples.csv' not found. Cannot save category examples map.")

print(f"Best trained model saved as {model_filename}")
print("Encoders (onehot_encoder_X.pkl, label_encoder_y.pkl) saved.")
print("Category examples map (category_examples_map.pkl) saved (if file found).\n")
print("\n" + "="*50 + "\n")

# --- 5. Example Prediction Function (using the saved components) ---

print("Example of how to use the saved model for new predictions:")

def predict_exercise_category_from_saved(age_group, medical_condition, flexibility_level, pain_level):
    try:
        loaded_model = joblib.load('random_forest_model_tuned.pkl')
        loaded_encoder_X = joblib.load('onehot_encoder_X.pkl')
        loaded_encoder_y = joblib.load('label_encoder_y.pkl')
        loaded_category_to_examples = joblib.load('category_examples_map.pkl')
    except FileNotFoundError as e:
        return f"Error loading prediction components: {e}. Make sure all .pkl and .csv files are present."


    new_user_data = pd.DataFrame([[age_group, medical_condition, flexibility_level, pain_level]],
                                 columns=['Age Group', 'Medical Condition', 'Flexibility Level', 'Pain Level'])

    new_user_encoded = loaded_encoder_X.transform(new_user_data)
    new_user_encoded_df = pd.DataFrame(new_user_encoded, columns=loaded_encoder_X.get_feature_names_out(new_user_data.columns))

    predicted_category_encoded = loaded_model.predict(new_user_encoded_df)
    predicted_category_name = loaded_encoder_y.inverse_transform(predicted_category_encoded)[0]

    examples = loaded_category_to_examples.get(predicted_category_name, "No examples found for this category.")

    return predicted_category_name, examples

# Test with your example user
print("\n--- Testing Prediction with Saved Model ---")
predicted_category, examples_for_category = predict_exercise_category_from_saved(
    age_group='Adult',
    medical_condition='Rehab',
    flexibility_level='Poor',
    pain_level='Severe'
)

print(f"\nFor the new user profile:")
print(f"  Age Group: Adult")
print(f"  Medical Condition: Rehab")
print(f"  Flexibility Level: Poor")
print(f"  Pain Level: Severe")
print(f"🎯 Predicted Exercise Category: {predicted_category}")
print(f"Example Exercises: {examples_for_category}")

# Another example
predicted_category2, examples_for_category2 = predict_exercise_category_from_saved(
    age_group='Senior',
    medical_condition='Rehab',
    flexibility_level='Moderate',
    pain_level='Moderate'
)
print(f"\nFor the new user profile:")
print(f"  Age Group: Senior")
print(f"  Medical Condition: Rehab")
print(f"  Flexibility Level: Moderate")
print(f"  Pain Level: Moderate")
print(f"🎯 Predicted Exercise Category: {predicted_category2}")
print(f"Example Exercises: {examples_for_category2}")

Data preparation complete. Shapes of training and testing sets:
X_train shape: (8000, 13)
X_test shape: (2000, 13)
y_train shape: (8000,)
y_test shape: (2000,)


Starting Hyperparameter Tuning for RandomForestClassifier...
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Hyperparameter tuning complete.
Best parameters found: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best cross-validation accuracy: 1.0000


Evaluating the model with best parameters on the test set...
Test Accuracy with Best Model: 1.0000

Classification Report with Best Model:
                                         precision    recall  f1-score   support

   Advanced Rehab & High-Level Function       1.00      1.00      1.00        55
        Advanced Strength & Performance       1.00      1.00      1.00        72
         Basic Flexibility & Low-Impact       1.00      1.00      1.00       114
          Early Rehab & Pain Management     